# 06. End-to-End Recommendation Pipeline & Evaluation

Notebook này kết nối tất cả các Stage đơn lẻ lại thành một hệ thống gợi ý hoàn chỉnh 3 lớp (Retrieval -> Ranking -> Re-ranking) và tiến hành đánh giá toàn diện bằng các chỉ số Accuracy & Beyond-Accuracy.

---

### Cấu trúc luồng chạy thử nghiệm:
1.  **Stage 1: Retrieval**: Gọi cả hai mô hình **iALS** (Collaborative) và **TF-IDF Cosine** (Content-based) để lấy ra Top-150 candidates mỗi bên, gộp lại (Union) được khoảng ~250 candidates.
2.  **Stage 2: Ranking**: Dùng mô hình **LightGBM LGBMRanker** để chấm điểm chi tiết cho ~250 candidates của User.
3.  **Stage 3: Re-ranking**: Áp dụng **MMR (lambda=0.7)** để chọn ra Top-10 phim đa dạng và chất lượng nhất đưa tới client.
4.  **Evaluation**: Đánh giá dựa trên tập Test LOO bằng các chỉ số: **Hit Ratio@10 (HR@10)**, **NDCG@10**, **Diversity**, **Coverage**, **Novelty**.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import pickle

sys.path.append(os.path.abspath('..'))
from recsys_utils import evaluate_implicit_loo, calculate_beyond_accuracy_metrics

# Load models
with open("models/als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
    
with open("models/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)
    
with open("models/tfidf_vectorizer.pkl", "rb") as f:
    tfidf_vectorizer = pickle.load(f)
    
with open("models/lgb_ranker.pkl", "rb") as f:
    lgb_ranker = pickle.load(f)

# Load data
train_ratings = pd.read_csv("processed_data/train_ratings.csv")
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
users_df = pd.read_csv(os.path.join("..", "..", "data", "simulator", "sim_users.csv"))

with open("processed_data/test_data.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("processed_data/id_mappings.pkl", "rb") as f:
    user_to_idx, movie_to_idx, idx_to_movie = pickle.load(f)
    
with open("processed_data/user_item_matrix.pkl", "rb") as f:
    user_item_matrix = pickle.load(f)


In [2]:
# 1. Định nghĩa End-to-End Pipeline
movies_df['genres'] = movies_df['genres'].fillna('')
movies_indexed_df = movies_df.set_index('movieId')
users_indexed_df = users_df.set_index('user_id')

def end_to_end_recommend(user_id, top_k=10, lambda_param=0.7):
    # --- STAGE 1: RETRIEVAL ---
    liked_movies = train_ratings[train_ratings['userId'] == user_id]['movieId'].tolist()
    liked_set = set(liked_movies)
    
    cb_candidates = []
    sim_scores = None
    liked_idx = movies_df[movies_df['movieId'].isin(liked_movies)].index.tolist()
    if liked_idx:
        from sklearn.metrics.pairwise import cosine_similarity
        sim_scores = cosine_similarity(tfidf_matrix[liked_idx], tfidf_matrix).mean(axis=0)
        
        # Lọc bỏ phim đã thích bằng cách giảm similarity score của chúng về tối thiểu (-1.0)
        for lid in liked_movies:
            if lid in movie_to_idx:
                sim_scores[movie_to_idx[lid]] = -1.0
                
        # Tối ưu hóa: Trích xuất top 100 chỉ số tương đồng lớn nhất bằng np.argpartition (nhanh hơn np.argsort toàn bộ)
        top_n_idx = min(100, len(sim_scores))
        partitioned_idx = np.argpartition(-sim_scores, top_n_idx)[:top_n_idx]
        sorted_cb_idx = partitioned_idx[np.argsort(-sim_scores[partitioned_idx])]
        cb_candidates = movies_df.iloc[sorted_cb_idx]['movieId'].tolist()
        
    u_idx = user_to_idx.get(user_id, None)
    als_candidates = []
    if u_idx is not None:
        ids, _ = als_model.recommend(u_idx, user_item_matrix[u_idx], N=100)
        # Loại bỏ các phim đã thích khỏi danh sách đề xuất của iALS
        als_candidates = [idx_to_movie[i] for i in ids if i in idx_to_movie and idx_to_movie[i] not in liked_set]
        
    candidates = list(set(cb_candidates + als_candidates))
    if not candidates:
        candidates = movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(100).tolist()
        candidates = [cid for cid in candidates if cid not in liked_set]
        
    # --- STAGE 2: RANKING ---
    features = []
    valid_candidates = []
    
    als_user_factors = als_model.user_factors
    als_item_factors = als_model.item_factors
    
    for mid in candidates:
        if mid not in movies_indexed_df.index:
            continue
        valid_candidates.append(mid)
        movie = movies_indexed_df.loc[mid]
        user = users_indexed_df.loc[user_id]
        
        favorite_genres = set(user['favorite_genres'].split('|'))
        movie_genres = set(movie['genres'].split('|'))
        genre_overlap = len(favorite_genres.intersection(movie_genres))
        
        try:
            release_year = int(str(movie['release_date'])[:4])
        except:
            release_year = 2010
            
        # --- Tính đặc trưng Retrieval score ---
        u_idx = user_to_idx.get(user_id, None)
        m_idx = movie_to_idx.get(mid, None)
        
        als_score = als_user_factors[u_idx].dot(als_item_factors[m_idx]) if (u_idx is not None and m_idx is not None) else 0.0
        cb_score = sim_scores[m_idx] if (sim_scores is not None and m_idx is not None) else 0.0
            
        features.append({
            'popularity': movie['popularity'],
            'vote_average': movie['vote_average'],
            'genre_overlap': genre_overlap,
            'release_year': release_year,
            'user_activity': user['activity_level'],
            'user_bias': user['user_bias'],
            'als_score': als_score,
            'cb_score': cb_score
        })
        
    X_pred = pd.DataFrame(features)
    scores = lgb_ranker.predict(X_pred)
    
    candidate_scores = list(zip(valid_candidates, scores))
    candidate_scores.sort(key=lambda x: x[1], reverse=True)
    
    # --- STAGE 3: RE-RANKING (MMR) ---
    from sklearn.metrics.pairwise import cosine_similarity
    
    final_recs = []
    if candidate_scores:
        candidates_ids = [item[0] for item in candidate_scores]
        scores_arr = np.array([item[1] for item in candidate_scores])
        
        if scores_arr.max() != scores_arr.min():
            scores_norm = (scores_arr - scores_arr.min()) / (scores_arr.max() - scores_arr.min())
        else:
            scores_norm = np.ones_like(scores_arr)
            
        selected_items = []
        unselected_indices = list(range(len(candidates_ids)))
        
        first_choice = np.argmax(scores_norm)
        selected_items.append(candidates_ids[first_choice])
        unselected_indices.remove(first_choice)
        
        while len(selected_items) < top_k and unselected_indices:
            best_mmr = -1
            best_candidate_idx = -1
            
            selected_matrix_indices = [movie_to_idx[mid] for mid in selected_items if mid in movie_to_idx]
            if not selected_matrix_indices:
                break
                
            selected_vectors = tfidf_matrix[selected_matrix_indices]
            
            for idx in unselected_indices:
                candidate_id = candidates_ids[idx]
                candidate_matrix_idx = movie_to_idx.get(candidate_id, None)
                if candidate_matrix_idx is None:
                    continue
                candidate_vector = tfidf_matrix[candidate_matrix_idx]
                
                sim_with_selected = cosine_similarity(candidate_vector, selected_vectors).max()
                mmr_val = lambda_param * scores_norm[idx] - (1 - lambda_param) * sim_with_selected
                
                if mmr_val > best_mmr:
                    best_mmr = mmr_val
                    best_candidate_idx = idx
                    
            if best_candidate_idx == -1:
                break
            selected_items.append(candidates_ids[best_candidate_idx])
            unselected_indices.remove(best_candidate_idx)
        final_recs = selected_items
        
    return final_recs

print("Pipeline End-to-End đã xây dựng xong!")


Pipeline End-to-End đã xây dựng xong!


In [3]:
# 2. Đánh giá thử nghiệm hệ thống
predictions_dict = {}
pipeline_recs = {}

print("Bắt đầu đánh giá Pipeline trên 200 users từ tập test LOO...")
for u, pos_item, neg_items in test_data:
    u = int(u)
    pos_item = int(pos_item)
    neg_items = [int(x) for x in neg_items]
    
    recs = end_to_end_recommend(u, top_k=10, lambda_param=0.7)
    pipeline_recs[u] = recs
    
    items = [pos_item] + neg_items
    
    user_preds = []
    for item in items:
        if item in recs:
            score = 10 - recs.index(item)
        else:
            score = 0
        user_preds.append((item, score, item == pos_item))
    predictions_dict[u] = user_preds

# 3. Tính toán các metrics
hr, ndcg, mrr = evaluate_implicit_loo(predictions_dict, k=10)
div, nov, cov = calculate_beyond_accuracy_metrics(
    pipeline_recs, train_ratings, movies_df, movie_features=None, k=10, item_col='movieId'
)

print("\n=== KẾT QUẢ ĐÁNH GIÁ END-TO-END PIPELINE (THUẦN ML) ===")
print(f"Hit Ratio@10 (HR@10):  {hr:.4f}")
print(f"NDCG@10:               {ndcg:.4f}")
print(f"Mean Reciprocal Rank:  {mrr:.4f}")
print(f"Diversity@10:          {div:.4f}")
print(f"Novelty@10:            {nov:.4f}")
print(f"Coverage@10:           {cov:.4f}")


Bắt đầu đánh giá Pipeline trên 200 users từ tập test LOO...



=== KẾT QUẢ ĐÁNH GIÁ END-TO-END PIPELINE (THUẦN ML) ===
Hit Ratio@10 (HR@10):  1.0000
NDCG@10:               0.9427
Mean Reciprocal Rank:  0.9225
Diversity@10:          0.8168
Novelty@10:            10.6830
Coverage@10:           0.1210


## Kết luận và Giải pháp đề xuất cho dự án

*   **Hiệu năng vượt trội**: Pipeline thuần ML kết hợp Stage 1 (iALS + CB) -> Stage 2 (LightGBM) -> Stage 3 (MMR) mang lại kết quả chất lượng vượt trội nhờ khả năng tối ưu hóa đa lớp.
*   **Cold Start được xử lý**:
    *   Nhờ nhánh **Content-Based TF-IDF** ở Stage 1, các phim mới 2026 hoàn toàn có thể được chọn làm ứng viên và đưa vào Ranker để gợi ý ngay lập tức.
*   **Khả năng phân tách & diễn giải (Interpretability)**:
    *   LightGBM cho phép phân tích Feature Importance để giải thích lý do xếp hạng.
    *   MMR kiểm soát trực tiếp độ đa dạng của danh sách phim để đáp ứng thị huớng phong phú của người dùng.
